# APTOS — Chia dữ liệu 5-fold đơn giản (chỉ trùng tuyệt đối)

**Mục tiêu:** tạo bộ chia dùng chung cho mọi phương pháp, khoảng **70% Train / 10% Validation / 20% Test** mỗi lượt.

Notebook chỉ cần **train.csv và train_images của APTOS 2019**. Không cần kết quả phân tích cũ, không dùng test gốc không nhãn, không xét pHash/dHash/SSIM hay ảnh gần trùng, không huấn luyện mô hình.

Luồng: **đọc pixel gốc → gom SHA-256 → loại cụm mâu thuẫn → nhóm kích thước → chia 5 lượt → kiểm tra → xuất 5 file**.

Đây là giao thức **chỉ kiểm soát trùng tuyệt đối theo yêu cầu mới**; không tuyên bố đã hoàn thành phần gần trùng của Mục 5.3, hoặc loại hết rò rỉ theo bệnh nhân (không có mã bệnh nhân).

**Cách chạy trên Kaggle:** thêm bộ dữ liệu APTOS, kiểm tra đường dẫn ở bước 1, chọn CPU rồi Run All. Không cần GPU. Các file kết quả nằm trong một thư mục mới ở `/kaggle/working`.


## 1. Thư viện và cấu hình
Dùng scikit-learn 1.8.0 để tái lập kết quả. Bản 1.8 đã sửa lỗi `shuffle` của StratifiedGroupKFold: [ghi chú chính thức](https://sklearn.org/1.8/whats_new/v1.8.html).

Nếu môi trường cũ hơn 1.8, bỏ dấu `#` ở dòng cài đặt, chạy cell, **khởi động lại phiên** rồi Run All. Những lần chạy cần tái lập chính xác nên dùng cùng phiên bản thư viện.


In [ ]:
%pip install -q scikit-learn==1.8.0

In [ ]:
# Chỉ bỏ dấu # nếu cần cài thư viện; sau đó khởi động lại phiên.


from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from itertools import combinations
import hashlib
import json
import math
import platform
import re

import numpy as np
import pandas as pd
import sklearn
import PIL
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from IPython.display import display, FileLink

version = tuple(map(int, re.match(r"(\d+)\.(\d+)", sklearn.__version__).groups()))
if version < (1, 8):
    raise RuntimeError("Cần scikit-learn >= 1.8.0. Cài bản 1.8.0, khởi động lại phiên rồi Run All.")

DATA_ROOT = Path('/kaggle/input/competitions/aptos2019-blindness-detection')
TRAIN_CSV = DATA_ROOT / 'train.csv'
IMAGE_DIR = DATA_ROOT / 'train_images'
OUTPUT_ROOT = Path('/kaggle/working')

SEED = 42
N_FOLDS = 5
VAL_FOLDS = 8
MIN_SIZE_IMAGES = 30       # Kích thước có < 30 ảnh gốc → other
MIN_STRATUM_IMAGES = 10    # Ô grade × nhóm có < 10 ảnh → gộp
EXPECTED_IMAGES = 3662
print('scikit-learn:', sklearn.__version__)


## 2. Đọc nhãn, kích thước và SHA-256 trên pixel gốc
Không crop, resize hay chuẩn hóa trước khi băm. Kích thước lấy trực tiếp từ ảnh gốc.

Để khớp cách băm của đợt phân tích cũ bằng OpenCV, mảng RGB được đổi thứ tự kênh thành BGR; SHA-256 gồm hình dạng mảng, kiểu dữ liệu và byte pixel. Hai file khác cách nén nhưng cùng pixel vẫn thuộc cùng cụm.

Quét tuần tự để hạn chế RAM; chỉ in tiến độ mỗi khoảng 10%. Nếu thiếu/hỏng ảnh, notebook dừng và báo tên file, không tự bỏ ảnh.


In [ ]:
labels = pd.read_csv(TRAIN_CSV, dtype={'id_code': str})
assert {'id_code', 'diagnosis'} <= set(labels.columns), 'Thiếu cột id_code hoặc diagnosis.'
assert len(labels) == EXPECTED_IMAGES, f'Cần {EXPECTED_IMAGES} ảnh; đang có {len(labels)}.'
assert labels.id_code.notna().all() and labels.id_code.is_unique, 'ID rỗng hoặc trùng.'
assert labels.id_code.str.fullmatch(r'[A-Za-z0-9_-]+').all(), 'ID không hợp lệ.'
assert labels.diagnosis.isin(range(5)).all(), 'Nhãn phải là số nguyên từ 0 đến 4.'
labels = labels.rename(columns={'diagnosis': 'grade'}).sort_values('id_code').reset_index(drop=True)
labels['grade'] = labels.grade.astype(int)

def read_image_info(path):
    with Image.open(path) as image:
        # APTOS gốc dùng RGB 8-bit. Dừng nếu khác để tránh chuyển đổi âm thầm.
        if image.mode != 'RGB':
            raise ValueError(f'{path.name}: mode={image.mode}, cần ảnh RGB gốc.')
        rgb = np.asarray(image)
        if rgb.dtype != np.uint8:
            raise ValueError(f'{path.name}: kiểu pixel không phải uint8.')
        bgr = np.ascontiguousarray(rgb[:, :, ::-1])
        h = hashlib.sha256()
        h.update(str((bgr.shape, str(bgr.dtype))).encode())
        h.update(bgr.tobytes())
        return {'width': image.width, 'height': image.height,
                'file_bytes': path.stat().st_size, 'pixel_sha256': h.hexdigest()}

rows = []
progress_step = math.ceil(len(labels) / 10)
for i, row in enumerate(labels.itertuples(index=False), start=1):
    path = IMAGE_DIR / f'{row.id_code}.png'
    if not path.is_file():
        raise FileNotFoundError(path)
    rows.append({'id_code': row.id_code, 'grade': row.grade, **read_image_info(path)})
    if i % progress_step == 0 or i == len(labels):
        print(f'Đã đọc {i:,}/{len(labels):,} ảnh ({i / len(labels):.0%})')
metadata = pd.DataFrame(rows)
print('Đã đọc đủ ảnh. Số nội dung pixel khác nhau:', metadata.pixel_sha256.nunique())


## 3. Gom cụm và xử lý mâu thuẫn nhãn
Vì chỉ xét trùng tuyệt đối, các ảnh có cùng SHA-256 chính là một thành phần liên thông; không cần thuật toán đồ thị riêng. Dùng đầy đủ mã băm làm `cluster_id` để ID ổn định giữa các lần chạy.

**Quy tắc cố định:** nếu một cụm có nhiều grade, loại **toàn bộ cụm khỏi cả Train, Validation và Test**, vì chưa có nhãn chuyên gia để quyết định nhãn đúng. Không sửa nhãn theo đa số. Các bản trùng cùng nhãn được giữ và luôn đi cùng nhau trong từng lượt.

`clusters.csv` vẫn chứa toàn bộ 3.662 ID để truy vết. Số 30 cụm / 62 ảnh từ báo cáo trước chỉ là mốc đối chiếu; notebook tính lại từ dữ liệu thực tế.


In [ ]:
metadata['cluster_id'] = 'sha256_' + metadata.pixel_sha256
conflict_ids = set(metadata.groupby('cluster_id').grade.nunique().loc[lambda s: s > 1].index)
metadata['excluded'] = metadata.cluster_id.isin(conflict_ids)

conflict_rows = []
for cluster_id, part in metadata[metadata.excluded].groupby('cluster_id', sort=True):
    counts = part.grade.value_counts()
    conflict_rows.append({
        'cluster_id': cluster_id,
        'n_images': len(part),
        'id_codes': ';'.join(sorted(part.id_code)),
        'observed_grades': ';'.join(map(str, sorted(part.grade.unique()))),
        **{f'grade_{g}_count': int(counts.get(g, 0)) for g in range(5)},
        'decision': 'exclude_from_train_val_test',
        'reason': 'Cùng pixel nhưng khác nhãn; chưa có nhãn chuyên gia xác nhận.'
    })
conflict_columns = ['cluster_id', 'n_images', 'id_codes', 'observed_grades'] + [
    f'grade_{g}_count' for g in range(5)] + ['decision', 'reason']
conflicts = pd.DataFrame(conflict_rows, columns=conflict_columns)
print(f'Loại {len(conflict_ids)} cụm, {metadata.excluded.sum()} ảnh mâu thuẫn nhãn.')
display(metadata.groupby(['grade', 'excluded']).size().unstack(fill_value=0)
        .rename(columns={False: 'giữ', True: 'loại'}))


## 4. Nhóm thu nhận theo kích thước gốc
Đếm kích thước trên **toàn bộ 3.662 ảnh có nhãn trước khi loại mâu thuẫn**. Kích thước có ít nhất 30 ảnh giữ tên `WxH`; dưới 30 ảnh gộp vào `other` (khác). Quy tắc này được giữ chung cho cả 5 lượt.

Nhóm kích thước chỉ là đại diện cho điều kiện thu nhận, không phải mã thiết bị đã xác minh.


In [ ]:
metadata['original_size'] = metadata.width.astype(str) + 'x' + metadata.height.astype(str)
size_counts = metadata.original_size.value_counts()
metadata['acquisition_group'] = metadata.original_size.where(
    metadata.original_size.map(size_counts) >= MIN_SIZE_IMAGES, 'other')

eligible = metadata[~metadata.excluded].sort_values('id_code').reset_index(drop=True)
assert len(eligible) > 0 and set(eligible.grade) == set(range(5)), 'Dữ liệu hợp lệ thiếu lớp.'
print('Ảnh đủ điều kiện:', len(eligible))
display(pd.crosstab(eligible.grade, eligible.acquisition_group))


## 5. Tạo nhãn phân tầng, chỉ gộp các ô cần thiết
Trong từng grade:
1. Giữ riêng ô có **ít nhất 10 ảnh và số cụm ít nhất bằng số fold**.
2. Gộp các ô còn lại thành một nhóm hiếm trong cùng grade.
3. Nếu nhóm hiếm vẫn chưa đủ: ghép thêm ô `other` nếu có; nếu không, ghép ô đủ điều kiện nhỏ nhất (hòa thì theo tên). Lặp đến khi đủ.
4. Các ô lớn không tham gia ghép vẫn giữ riêng. Nếu cả grade không đủ điều kiện, dừng và báo lỗi.

Quy tắc được ấn định trước, không thử nhiều seed hay chọn bộ chia theo kết quả mô hình. Áp dụng lại trên phần Train+Validation của từng lượt để lấy Validation. Nhãn phân tầng có thể gộp, nhưng **grade và acquisition_group gốc không đổi**.


In [ ]:
def make_strata(frame, n_splits):
    strata = pd.Series(index=frame.index, dtype='object')
    mapping = []
    for grade, part in frame.groupby('grade', sort=True):
        stats = part.groupby('acquisition_group').agg(
            n_images=('id_code', 'size'), n_clusters=('cluster_id', 'nunique'))
        if len(part) < MIN_STRATUM_IMAGES or part.cluster_id.nunique() < n_splits:
            raise ValueError(f'Grade {grade} không đủ ảnh/cụm cho {n_splits} fold.')
        rare = stats.index[(stats.n_images < MIN_STRATUM_IMAGES) |
                           (stats.n_clusters < n_splits)].tolist()
        pool = set(rare)
        retained = set(stats.index) - pool
        absorbed = []
        while pool:
            pooled_rows = part[part.acquisition_group.isin(pool)]
            if len(pooled_rows) >= MIN_STRATUM_IMAGES and pooled_rows.cluster_id.nunique() >= n_splits:
                break
            if not retained:
                raise ValueError(f'Không tạo được nhóm phân tầng cho grade {grade}.')
            chosen = ('other' if 'other' in retained else
                      min(retained, key=lambda name: (int(stats.loc[name, 'n_images']), name)))
            retained.remove(chosen)
            pool.add(chosen)
            absorbed.append(chosen)
        for group in stats.index:
            label = f'{grade}|pooled' if group in pool else f'{grade}|{group}'
            strata.loc[part.index[part.acquisition_group.eq(group)]] = label
            mapping.append({
                'grade': int(grade), 'acquisition_group': group, 'stratum': label,
                'n_images': int(stats.loc[group, 'n_images']),
                'n_clusters': int(stats.loc[group, 'n_clusters']),
                'action': ('pooled_rare' if group in rare else
                           'absorbed_to_support_rare' if group in absorbed else 'kept_separate')
            })
    assert strata.notna().all()
    check = frame.assign(stratum=strata).groupby('stratum').agg(
        n_images=('id_code', 'size'), n_clusters=('cluster_id', 'nunique'))
    assert (check.n_images >= MIN_STRATUM_IMAGES).all()
    assert (check.n_clusters >= n_splits).all()
    return strata, mapping, check.reset_index().to_dict('records')

outer_y, outer_mapping, outer_counts = make_strata(eligible, N_FOLDS)
print('Các ô được gộp ở vòng chia Test:')
display(pd.DataFrame(outer_mapping).loc[lambda d: d.action != 'kept_separate'])


## 6. Chia 5 lượt Train / Validation / Test
- Vòng ngoài: StratifiedGroupKFold 5 fold, shuffle=True, seed=42; mỗi fold làm Test một lần.
- Trong 80% còn lại: StratifiedGroupKFold 8 fold, cùng seed; luôn chọn fold đầu tiên làm Validation.
- Train khoảng 70%, Validation 10%, Test 20%. Thuật toán cố cân bằng phân tầng trong giới hạn ràng buộc cụm; tỷ lệ không bắt buộc chính xác tuyệt đối.

Số thứ tự `fold` trong JSON là **0 đến 4**. Mỗi phương pháp phải khởi tạo mô hình mới cho từng lượt, dùng Validation chọn checkpoint/siêu tham số và chỉ dùng Test để đánh giá.


In [ ]:
outer_cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds, inner_reports, overview = [], [], []
for fold, (remaining_idx, test_idx) in enumerate(
        outer_cv.split(eligible, outer_y, eligible.cluster_id)):
    remaining = eligible.iloc[remaining_idx].reset_index(drop=True)
    inner_y, inner_mapping, inner_counts = make_strata(remaining, VAL_FOLDS)
    inner_cv = StratifiedGroupKFold(n_splits=VAL_FOLDS, shuffle=True, random_state=SEED)
    train_idx, val_idx = next(inner_cv.split(remaining, inner_y, remaining.cluster_id))
    subsets = {'train': remaining.iloc[train_idx],
               'val': remaining.iloc[val_idx], 'test': eligible.iloc[test_idx]}
    folds.append({'fold': fold, **{name: sorted(part.id_code.tolist())
                                  for name, part in subsets.items()}})
    inner_reports.append({'fold': fold, 'mapping': inner_mapping, 'strata_counts': inner_counts})
    for name, part in subsets.items():
        overview.append({'fold': fold, 'subset': name, 'n_images': len(part),
                         'n_clusters': part.cluster_id.nunique(),
                         'percent_of_eligible': round(100 * len(part) / len(eligible), 4)})
display(pd.DataFrame(overview))


## 7. Kiểm tra và gom thống kê vào một báo cáo
Kiểm tra trong từng lượt: ID/cụm/SHA-256 không chồng lấp; đủ mọi ảnh hợp lệ; không có ảnh mâu thuẫn; có đủ 5 grade. Kiểm tra xuyên 5 lượt: mỗi ảnh và mỗi cụm hợp lệ làm Test đúng một lần.

Báo cáo chứa số ảnh, số cụm, phân bố grade, nhóm kích thước và bảng grade × nhóm cho cả 15 tập. Cả các ô có 0 ảnh cũng được ghi lại. Chênh lệch phân bố được tính bằng **điểm phần trăm** so với toàn bộ dữ liệu hợp lệ.

Không xuất kết quả nếu kiểm tra bắt buộc thất bại. Kết luận không rò rỉ chỉ áp dụng cho cụm **trùng tuyệt đối đã phát hiện**, không bao gồm gần trùng.


In [ ]:
lookup = eligible.set_index('id_code', drop=False)
all_ids = set(eligible.id_code)
excluded_ids = set(metadata.loc[metadata.excluded, 'id_code'])
acq_names = sorted(metadata.acquisition_group.unique())

def distribution(frame):
    n = len(frame)
    gc = {str(g): int((frame.grade == g).sum()) for g in range(5)}
    ac = {a: int((frame.acquisition_group == a).sum()) for a in acq_names}
    joint = {str(g): {a: int(((frame.grade == g) & (frame.acquisition_group == a)).sum())
                      for a in acq_names} for g in range(5)}
    return {'n_images': n, 'n_clusters': int(frame.cluster_id.nunique()),
            'grade_counts': gc, 'acquisition_counts': ac, 'grade_acquisition_counts': joint,
            'grade_percent': {g: round(100*v/n, 4) if n else 0.0 for g,v in gc.items()},
            'acquisition_percent': {a: round(100*v/n, 4) if n else 0.0 for a,v in ac.items()}}

baseline = distribution(eligible)
audits, distributions = [], []
test_id_counts, test_cluster_counts = Counter(), Counter()
for fold in folds:
    parts = {name: lookup.loc[fold[name]].copy() for name in ['train', 'val', 'test']}
    checks = {
        'unique_ids_within_subsets': all(len(fold[n]) == len(set(fold[n])) for n in parts),
        'complete_eligible_coverage': set().union(*(set(p.id_code) for p in parts.values())) == all_ids,
        'excluded_absent': all(not (set(p.id_code) & excluded_ids) for p in parts.values()),
        'all_five_grades': all(set(p.grade) == set(range(5)) for p in parts.values())
    }
    for a, b in combinations(parts, 2):
        for col in ['id_code', 'cluster_id', 'pixel_sha256']:
            checks[f'{a}_{b}_{col}_disjoint'] = set(parts[a][col]).isdisjoint(parts[b][col])
    if not all(checks.values()):
        raise AssertionError(f'Fold {fold["fold"]} kiểm tra thất bại: {checks}')
    audits.append({'fold': fold['fold'], 'passed': True, 'checks': checks})
    test_id_counts.update(parts['test'].id_code)
    test_cluster_counts.update(set(parts['test'].cluster_id))
    for name, part in parts.items():
        info = distribution(part)
        info['grade_deviation_pp'] = {
            g: round(100 * info['grade_counts'][g] / len(part) -
                     100 * baseline['grade_counts'][g] / len(eligible), 4) for g in baseline['grade_counts']}
        info['acquisition_deviation_pp'] = {
            a: round(100 * info['acquisition_counts'][a] / len(part) -
                     100 * baseline['acquisition_counts'][a] / len(eligible), 4) for a in acq_names}
        distributions.append({'fold': fold['fold'], 'subset': name, **info})

global_checks = {
    'each_eligible_image_test_once': set(test_id_counts) == all_ids and all(v == 1 for v in test_id_counts.values()),
    'each_eligible_cluster_test_once': set(test_cluster_counts) == set(eligible.cluster_id) and all(v == 1 for v in test_cluster_counts.values()),
    'all_input_ids_accounted_for': all_ids.isdisjoint(excluded_ids) and (all_ids | excluded_ids) == set(metadata.id_code),
    'cluster_has_one_pixel_hash': bool(metadata.groupby('cluster_id').pixel_sha256.nunique().eq(1).all()),
    'pixel_hash_has_one_cluster': bool(metadata.groupby('pixel_sha256').cluster_id.nunique().eq(1).all())
}
assert all(global_checks.values()), global_checks
print('PASS: 5 lượt không chồng lấp ID/cụm/pixel, không có ảnh mâu thuẫn; mỗi ảnh Test đúng 1 lần.')


## 8. Xuất đúng 5 file
| File | Nội dung |
|---|---|
| `clusters.csv` | `id_code, cluster_id`, toàn bộ ảnh có nhãn ban đầu |
| `acquisition_groups.csv` | ID, W, H, nhóm thu nhận; toàn bộ ảnh có nhãn ban đầu |
| `splits.json` | Cấu hình, ID bị loại và ID Train/Val/Test của 5 lượt |
| `conflict_decisions.csv` | Một dòng/cụm mâu thuẫn, ID thành viên, số ảnh theo grade, quyết định và lý do |
| `split_report.json` | Tổng hợp xử lý dữ liệu, phân bố, ánh xạ gộp ô, kiểm tra, phiên bản và mã băm truy vết |

Tạo thư mục kết quả mới mỗi lần chạy để tránh trộn với kết quả cũ. Không xuất ZIP hay bảng phụ. Dùng các liên kết dưới cell để tải từng file hoặc tải thư mục trong mục Output của Kaggle.


In [ ]:
config = {
    'dataset': 'APTOS 2019 original labeled train',
    'protocol': 'exact_duplicates_only', 'near_duplicates_considered': False,
    'seed': SEED, 'outer_n_splits': N_FOLDS, 'inner_n_splits': VAL_FOLDS,
    'algorithm': 'StratifiedGroupKFold', 'shuffle': True, 'group': 'cluster_id',
    'stratification': 'grade x acquisition_group with deterministic within-grade pooling',
    'validation_selection': 'first inner fold (index 0)',
    'target_ratios': {'train': 0.7, 'val': 0.1, 'test': 0.2},
    'minimum_original_size_count': MIN_SIZE_IMAGES,
    'minimum_stratum_images': MIN_STRATUM_IMAGES,
    'minimum_stratum_clusters': 'number of folds at the current split stage',
    'rare_pool_rule': 'pool rare cells within grade; if insufficient, absorb other first, then smallest adequate cell; ties by name',
    'conflict_policy': 'exclude_whole_cluster_from_all_subsets',
    'same_label_duplicates': 'retain_all_members_in_same_subset',
    'acquisition_count_basis': 'all original labeled images before exclusions',
    'pixel_hash_format': 'SHA256(str((BGR_array.shape, str(BGR_array.dtype))).encode() + contiguous_BGR_bytes)',
    'sklearn_version': sklearn.__version__
}

# Dấu vân tay chỉ phụ thuộc nội dung/nhãn, không phụ thuộc đường dẫn hoặc ngày chạy.
fingerprint_columns = ['id_code', 'grade', 'width', 'height', 'pixel_sha256']
fingerprint = hashlib.sha256(metadata[fingerprint_columns].sort_values('id_code')
    .to_csv(index=False, lineterminator='\n').encode('utf-8')).hexdigest()
created = datetime.now(timezone.utc)
splits = {
    'schema_version': 1, 'config': config, 'dataset_fingerprint_sha256': fingerprint,
    'n_original': len(metadata), 'n_eligible': len(eligible),
    'excluded_ids': sorted(excluded_ids), 'folds': folds
}
cluster_sizes = metadata.groupby('cluster_id').size()
report = {
    'schema_version': 1, 'created_at_utc': created.isoformat(),
    'scope': 'Chỉ trùng tuyệt đối trên pixel gốc; không xét gần trùng.',
    'status': 'passed_exact_duplicate_protocol', 'config': config,
    'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__, 'Pillow': PIL.__version__},
    'source': {'train_csv': str(TRAIN_CSV), 'image_dir': str(IMAGE_DIR),
               'train_csv_sha256': hashlib.sha256(TRAIN_CSV.read_bytes()).hexdigest(),
               'dataset_fingerprint_sha256': fingerprint},
    'original': distribution(metadata), 'eligible': baseline,
    'exact_duplicates': {
        'n_distinct_pixel_contents': int(len(cluster_sizes)),
        'n_duplicate_clusters': int((cluster_sizes > 1).sum()),
        'n_images_in_duplicate_clusters': int(cluster_sizes[cluster_sizes > 1].sum()),
        'n_redundant_copies_if_keep_one': int((cluster_sizes - 1).sum()),
        'note': 'Không xóa các bản trùng cùng nhãn; số bản dư chỉ là thống kê.'},
    'conflicts': {'n_clusters': len(conflict_ids), 'n_images_excluded': len(excluded_ids),
                  'excluded_by_grade': {str(g): int(((metadata.grade == g) & metadata.excluded).sum()) for g in range(5)},
                  'decisions': conflicts.to_dict('records')},
    'original_size_counts': {str(k): int(v) for k,v in size_counts.sort_index().items()},
    'stratification': {'outer': {'mapping': outer_mapping, 'strata_counts': outer_counts},
                       'inner_by_fold': inner_reports},
    'fold_overview': overview, 'fold_distributions': distributions,
    'max_absolute_deviation_pp': {
        'grade': max(abs(v) for d in distributions for v in d['grade_deviation_pp'].values()),
        'acquisition': max(abs(v) for d in distributions for v in d['acquisition_deviation_pp'].values())},
    'audit': {'all_passed': True, 'global_checks': global_checks, 'fold_checks': audits},
    'limitations': [
        'Không rà soát hay gom ảnh gần trùng; không tuyên bố hoàn tất phần gần trùng của Mục 5.3.',
        'Không có mã bệnh nhân nên chưa bảo đảm tách theo bệnh nhân.',
        'Không dùng test gốc không nhãn và không đánh giá chồng lấp với test gốc.',
        'Nhóm kích thước chỉ đại diện điều kiện thu nhận; không loại bỏ đường tắt thiết bị.',
        'Phân tầng là gần đúng dưới ràng buộc cụm; xem số ảnh và chênh lệch thực tế.',
        'Chưa chạy baseline kích thước, mô hình ảnh hoặc kiểm tra dấu vết thiết bị.'
    ]
}

OUT = OUTPUT_ROOT / ('aptos_split_exact_' + created.strftime('%Y%m%dT%H%M%S%fZ'))
OUT.mkdir(parents=True, exist_ok=False)
metadata[['id_code', 'cluster_id']].to_csv(OUT / 'clusters.csv', index=False)
metadata[['id_code', 'width', 'height', 'acquisition_group']].to_csv(
    OUT / 'acquisition_groups.csv', index=False)
conflicts.to_csv(OUT / 'conflict_decisions.csv', index=False)

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(type(value).__name__)

def save_json(name, value):
    (OUT / name).write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                     default=json_default, allow_nan=False), encoding='utf-8')

save_json('splits.json', splits)
report['output_sha256'] = {name: hashlib.sha256((OUT/name).read_bytes()).hexdigest()
    for name in ['clusters.csv', 'acquisition_groups.csv', 'conflict_decisions.csv', 'splits.json']}
save_json('split_report.json', report)
assert len(list(OUT.iterdir())) == 5
print('Đã lưu 5 file tại:', OUT)
for name in ['clusters.csv', 'acquisition_groups.csv', 'splits.json',
             'conflict_decisions.csv', 'split_report.json']:
    display(FileLink(str(OUT/name)))


## Dùng bộ chia khi huấn luyện
Đọc `splits.json`, chọn `folds[k]` rồi lấy các danh sách `train`, `val`, `test`. Ghép ID với `train.csv` để lấy grade và với `acquisition_groups.csv` để báo cáo theo nhóm. Mọi phương pháp dùng **cùng một bộ file đã chốt**, không tự chia lại.

Trong từng lượt: chỉ học từ Train; chọn checkpoint/ngưỡng dự đoán trên Validation; báo cáo Test sau khi chốt. Các thống kê dùng để học như trọng số lớp hoặc quy tắc lớp đa số theo kích thước phải tính từ Train của lượt đó.

**Cách mô tả trong báo cáo:** “Chúng tôi gom ảnh trùng tuyệt đối bằng SHA-256 trên pixel gốc, loại toàn bộ cụm mâu thuẫn nhãn, và dùng StratifiedGroupKFold 5 fold với seed 42. Phân tầng theo grade × nhóm kích thước, có gộp các ô hiếm trong cùng grade theo quy tắc cố định. Mỗi lượt lấy Validation bằng fold đầu tiên của vòng chia 8 fold trên phần còn lại. Nghiên cứu chưa xét ảnh gần trùng trong bước chia dữ liệu này.” Điền số lượng thực tế từ `split_report.json` sau khi chạy.
